In [1]:
# CUSTOM EXPERIMENT: Live Quantitative Pipeline (Rolling Origin: May 13, 2026)

import yfinance as yf
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

# 1. Define the Universe (Removed NGLFINE to prevent delisting/halt errors)
tickers = [
    'OFSS.NS', 'J&KBANK.NS', 'MANKIND.NS', 'SPARC.NS', 'ADANIPORTS.NS', 
    'ASIANENE.NS', 'LT.NS', 'CARYSIL.NS', 'SBILIFE.NS', 'SBIN.NS', 
    'BAJAJ-AUTO.NS', 'STARHEALTH.NS', 'NYKAA.NS', 'BHARATFORG.NS', 
    'POLICYBZR.NS', 'NATCOPHARM.NS', 'TATACOMM.NS', 'BANKBARODA.NS', 
    'ADANIGREEN.NS', 'TATASTEEL.NS', 'ADANIPOWER.NS', 'JSWSTEEL.NS', 
    'BAJAJCON.NS', 'GPIL.NS', 'IDEA.NS'
]

# 2. Fetch Training Data (Jan 2024 to May 13, 2026)
print("1. Ingesting market data up to May 13, 2026...")
train_data = yf.download(tickers, start="2024-01-01", end="2026-05-14")['Close']
train_data = train_data.dropna(axis=1, how='all').ffill().bfill()

# 3. Analyze Volatility, Trend, and Predict (May 14 & 15)
print("2. Analyzing Volatility, Seasonality, and Forecasting...")
analysis_results = []
capital_total = 1000000

for stock in train_data.columns:
    series = train_data[stock]
    
    # Calculate 60-Day Annualized Volatility
    recent_returns = series.tail(60).pct_change().dropna()
    volatility = recent_returns.std() * np.sqrt(252) 
    
    # Calculate Trend (50-Day vs 200-Day SMA)
    sma_50 = series.tail(50).mean()
    sma_200 = series.tail(200).mean()
    trend_status = "Bullish" if sma_50 > sma_200 else "Bearish"
    
    # Forecasting with Seasonality and Trend
    try:
        # seasonal_periods=5 accounts for weekly trading seasonality
        model = ExponentialSmoothing(series, trend='add', seasonal='add', seasonal_periods=5, initialization_method="estimated")
        fit_model = model.fit()
        forecast = fit_model.forecast(2)
        pred_day1, pred_day2 = forecast.iloc[0], forecast.iloc[1]
    except:
        # Fallback if seasonality fails on low-variance stocks
        model = ExponentialSmoothing(series, trend='add', initialization_method="estimated")
        fit_model = model.fit()
        forecast = fit_model.forecast(2)
        pred_day1, pred_day2 = forecast.iloc[0], forecast.iloc[1]
        
    last_price = series.iloc[-1]
    expected_return = (pred_day2 - last_price) / last_price
    
    analysis_results.append({
        'Stock': stock,
        'Last Price': last_price,
        'Volatility': volatility,
        'Trend': trend_status,
        'Expected Return': expected_return,
        'Pred Day 1': pred_day1,
        'Pred Day 2': pred_day2
    })

df_analysis = pd.DataFrame(analysis_results)

# 4. Dynamic Portfolio Allocation (Risk-Adjusted)
print("3. Executing Dynamic Risk-Adjusted Capital Allocation...")
# Calculate allocation score: Expected Return / Volatility. Floor negative returns at 0.
df_analysis['Score'] = df_analysis.apply(lambda x: max(0, x['Expected Return']) / x['Volatility'] if x['Volatility'] > 0 else 0, axis=1)

# Normalize scores to get percentage weights
total_score = df_analysis['Score'].sum()
if total_score > 0:
    df_analysis['Weight'] = df_analysis['Score'] / total_score
else:
    # Fallback to equal weight if model predicts market crash (all negative returns)
    df_analysis['Weight'] = 1.0 / len(df_analysis)

df_analysis['Allocated Capital (₹)'] = df_analysis['Weight'] * capital_total

# 5. Fetch Actual May Data and Evaluate
print("4. Fetching May 14-15 Realities and Calculating P&L...\n")
actual_data = yf.download(df_analysis['Stock'].tolist(), start="2026-05-14", end="2026-05-16")['Close']
actual_data = actual_data.ffill().bfill()

day1_actual_date, day2_actual_date = actual_data.index[0], actual_data.index[1]

evaluation_results = []
for index, row in df_analysis.iterrows():
    stock = row['Stock']
    capital = row['Allocated Capital (₹)']
    
    # Only trade if capital was allocated
    if capital > 0:
        actual_day1 = actual_data.loc[day1_actual_date, stock]
        actual_day2 = actual_data.loc[day2_actual_date, stock]
        
        shares = capital / actual_day1
        final_value = shares * actual_day2
        pnl = final_value - capital
        pct_return = ((actual_day2 - actual_day1) / actual_day1) * 100
        
        evaluation_results.append({
            'Stock': stock,
            'Weight (%)': round(row['Weight'] * 100, 2),
            'Allocated (₹)': round(capital, 2),
            'Pred Day 2': round(row['Pred Day 2'], 2),
            'Actual Day 2': round(actual_day2, 2),
            'Return (%)': round(pct_return, 2),
            'P&L (₹)': round(pnl, 2)
        })

df_eval = pd.DataFrame(evaluation_results)

# 6. Final Outputs
total_pnl = df_eval['P&L (₹)'].sum()
final_portfolio_value = capital_total + total_pnl
net_return = (total_pnl / capital_total) * 100

print("==================================================================")
print("DYNAMIC PORTFOLIO ALLOCATION & EXECUTION (MAY 14-15)")
print("==================================================================\n")
print(df_eval.sort_values(by='Weight (%)', ascending=False).to_string(index=False))

print("\n==================================================================")
print("FINAL PIPELINE PERFORMANCE SUMMARY")
print("==================================================================")
print(f"Initial Capital Deployed : ₹{capital_total:,.2f}")
print(f"Final Portfolio Value    : ₹{final_portfolio_value:,.2f}")
print(f"Total Profit / Loss      : ₹{total_pnl:,.2f}")
print(f"Net Portfolio Return     : {net_return:.2f}%")
print("==================================================================")

import os

# 1. Create a completely isolated directory structure
os.makedirs('../experiment_live/data', exist_ok=True)
os.makedirs('../experiment_live/reports', exist_ok=True)

# 2. Save the outputs into the isolated folders
df_analysis.to_csv('../experiment_live/reports/live_volatility_analysis.csv', index=False)
df_eval.to_csv('../experiment_live/reports/live_portfolio_evaluation.csv', index=False)

print("\nSuccess! Experiment data saved safely to the isolated '../experiment_live/' directory.")

1. Ingesting market data up to May 13, 2026...


[*********************100%***********************]  25 of 25 completed


2. Analyzing Volatility, Seasonality, and Forecasting...
3. Executing Dynamic Risk-Adjusted Capital Allocation...
4. Fetching May 14-15 Realities and Calculating P&L...



[*********************100%***********************]  25 of 25 completed


DYNAMIC PORTFOLIO ALLOCATION & EXECUTION (MAY 14-15)

        Stock  Weight (%)  Allocated (₹)  Pred Day 2  Actual Day 2  Return (%)  P&L (₹)
  BAJAJCON.NS       30.65      306515.76      530.37        537.45        3.66 11233.10
      OFSS.NS       16.37      163747.93     8960.10       9015.00        1.24  2032.02
 POLICYBZR.NS       11.60      115950.98     1639.61       1688.30        0.41   475.83
NATCOPHARM.NS       10.94      109438.16     1174.73       1199.10       -0.44  -481.59
     NYKAA.NS        7.92       79185.98      266.20        272.40       -0.22  -174.04
      SBIN.NS        7.22       72184.70      954.18        963.20        0.07    48.75
  ASIANENE.NS        6.48       64757.33      288.65        313.45        2.35  1522.46
ADANIPOWER.NS        5.62       56162.59      214.15        221.33       -1.41  -790.57
BHARATFORG.NS        2.44       24388.10     1946.93       1913.10       -1.84  -447.99
STARHEALTH.NS        0.77        7668.47      507.40        501.70